## NTT-Based Partitioned Circular Convolution!!!


In [152]:
import numpy as np

### Algorithms Parameters


In [153]:
N = 256 # number of points in the NTT
q = 7681 # prime modulus
W = 3844 # primitive root of unity for NTT modulo q
MB = 1*8*(2**10) # Number of random bits for total dataset generation
BLOCK_SIZE = 128 # Size of each block for partitioned convolution

### Generating 1MB Bitstream 

In [154]:
a = np.random.randint(0, 2, MB)
b = np.random.randint(0, 2, MB)
assert not np.array_equal(a, b), "Input arrays a and b should not be identical."
#print("Input array a:", a)
#print("Input array b:", b)

### Partitioned Convolution HW Golden Reference 

In [155]:
def partitioned_circular_convolution(A, B):

    N = len(A)

    assert len(B) == N, "A and B must have the same length"
    assert N % BLOCK_SIZE == 0, \
        "Input length must be divisible by 128"

    num_blocks = N // BLOCK_SIZE

    # Final circular-convolution output
    result = np.zeros(N, dtype=np.int64)

    for i in range(num_blocks):

        A_block = A[
            i * BLOCK_SIZE:
            (i + 1) * BLOCK_SIZE
        ]

        A_padded = np.zeros(N, dtype=np.int64)

        A_padded[:BLOCK_SIZE] = A_block

        for j in range(num_blocks):

            B_block = B[
                j * BLOCK_SIZE:
                (j + 1) * BLOCK_SIZE
            ]

            B_padded = np.zeros(N, dtype=np.int64)

            B_padded[:BLOCK_SIZE] = B_block

            C_block = np.convolve(
                A_block,
                B_block
            )

            output_start = (i + j) * BLOCK_SIZE

            for k in range(len(C_block)):

                output_index = (
                    output_start + k
                ) % N

                result[output_index] += C_block[k]

    return result

hw_golden_ref = partitioned_circular_convolution(a, b)

### Golden Convolution Reference

In [156]:
a_cap = np.fft.fft(a)
b_cap = np.fft.fft(b)
c_cap = a_cap * b_cap
c = np.fft.ifft(c_cap).real
sw_golden_ref = c.astype(np.int64)

In [157]:
sw_golden_ref

array([1898, 1905, 1939, ..., 1947, 1926, 1928])

In [158]:
assert np.array_equal(hw_golden_ref, sw_golden_ref), "Hardware and software golden references do not match."

### Bit Reversal Function

In [159]:
def bit_reverse_copy(a):
    n = len(a)
    if n == 0 or n & (n - 1):
        raise ValueError("The input length must be a positive power of two")

    result = np.zeros(n, dtype=np.int64)

    j = 0

    for i in range(n):
        result[j] = a[i]

        bit = n >> 1

        while j & bit:
            j ^= bit
            bit >>= 1

        j ^= bit

    return result

### NTT function

In [160]:
def iterative_ntt(a, q, W):
    n = len(a)

    # Step 1: bit-reverse the input for iterative radix-2 DIT ordering.
    a = bit_reverse_copy(a)

    # Step wise NTT solution
    stage_count = n.bit_length()
    b = np.zeros((stage_count, n), dtype=np.int64)
    b[0] = a.copy()
    # Step 2: iterative NTT
    length = 2
    stage_index = 1

    while length <= n:

        # Root used at this stage
        wlen = pow(W, n // length, q)

        # Process every block of this size
        for start in range(0, n, length):

            w = 1

            # Butterfly operations
            for j in range(start, start + length // 2):

                u = a[j]
                v = (w * a[j + length // 2]) % q

                a[j] = (u + v) % q
                a[j + length // 2] = (u - v) % q

                w = (w * wlen) % q

        length *= 2
        # storing the stage value 
        b[stage_index] = a.copy()
        stage_index += 1

    return a, b


### INTT Function 

In [161]:
def iterative_intt(a, q, W):
    n = len(a)

    # Inverse of the primitive root
    W_inv = pow(W, -1, q)

    # Step 1: bit-reverse the input for the matching inverse DIT pass.
    a = bit_reverse_copy(a)

    # Store intermediate stages
    stage_count = n.bit_length()
    b = np.zeros((stage_count, n), dtype=np.int64)
    b[0] = a.copy()

    # Step 2: iterative inverse NTT
    length = 2
    stage_index = 1

    while length <= n:

        # Inverse root used at this stage
        wlen = pow(W_inv, n // length, q)

        # Process every block
        for start in range(0, n, length):

            w = 1

            # Butterfly operations
            for j in range(start, start + length // 2):

                u = a[j]
                v = (w * a[j + length // 2]) % q

                a[j] = (u + v) % q
                a[j + length // 2] = (u - v) % q

                w = (w * wlen) % q

        length *= 2

        # Store stage
        b[stage_index] = a.copy()
        stage_index += 1

    # Step 3: multiply by n^(-1) mod q
    n_inv = pow(n, -1, q)

    for i in range(n):
        a[i] = (a[i] * n_inv) % q

    return a, b


### NTT DIF Function 

In [162]:
def iterative_ntt_dif(a, q, W):
    n = len(a)
    a = np.array(a, dtype=np.int64).copy()

    stage = 0
    length = n

    while length >= 2:
        half = length // 2
        wlen = pow(W, n // length, q)

        # -----------------------------------------
        # Print twiddle factors ONCE for this stage
        # -----------------------------------------
        twiddles = []
        w = 1

        for _ in range(half):
            twiddles.append(w)
            w = (w * wlen) % q

        print(f"\nStage {stage}")
        print("Length:", length)
        print("wlen:", wlen)
        print("Twiddle factors:", twiddles)

        # -----------------------------------------
        # DIF butterflies
        # -----------------------------------------
        for start in range(0, n, length):
            w = 1

            for j in range(start, start + half):
                u = a[j]
                v = a[j + half]

                a[j] = (u + v) % q
                a[j + half] = ((u - v) * w) % q

                w = (w * wlen) % q

        length //= 2
        stage += 1

    return a

### INTT DIT Function

In [163]:
def iterative_intt_dit(a, q, W):

    n = len(a)

    a = np.array(a, dtype=np.int64).copy()

    # Inverse root
    W_inv = pow(W, -1, q)

    length = 2

    while length <= n:

        half = length // 2

        # Inverse twiddle
        wlen = pow(
            W_inv,
            n // length,
            q
        )

        for start in range(0, n, length):

            w = 1

            for j in range(
                start,
                start + half
            ):

                u = a[j]

                v = (
                    w * a[j + half]
                ) % q

                # DIT inverse butterfly
                a[j] = (
                    u + v
                ) % q

                a[j + half] = (
                    u - v
                ) % q

                w = (
                    w * wlen
                ) % q

        length *= 2

    n_inv = pow(
        n,
        -1,
        q
    )

    a = (
        a * n_inv
    ) % q

    return a

### Partitioned Circular Convolution with Hardware mimiced NTT and INTT engine

In [164]:
BLOCK_SIZE = 128
NTT_SIZE = 256


def partitioned_circular_convolution_hw_mimic(A, B):

    N = len(A)

    assert len(B) == N, "A and B must have the same length"
    assert N % BLOCK_SIZE == 0, \
        "Input length must be divisible by BLOCK_SIZE"

    num_blocks = N // BLOCK_SIZE

    # Final N-point circular convolution
    result = np.zeros(N, dtype=np.int64)

    for block_i in range(num_blocks):

        # ====================================================
        # Get 128-point A block
        # ====================================================

        A_block = A[
            block_i * BLOCK_SIZE:
            (block_i + 1) * BLOCK_SIZE
        ]

        # ====================================================
        # Zero-pad A to 256 points
        # ====================================================

        A_padded = np.zeros(NTT_SIZE, dtype=np.int64)

        A_padded[:BLOCK_SIZE] = A_block

        # ====================================================
        # 256-point NTT
        # ====================================================

        A_ntt = iterative_ntt_dif(
            A_padded.copy(),
            q,
            W
        )

        # ====================================================
        # Process every B block
        # ====================================================

        for block_j in range(num_blocks):

            # ------------------------------------------------
            # Get 128-point B block
            # ------------------------------------------------

            B_block = B[
                block_j * BLOCK_SIZE:
                (block_j + 1) * BLOCK_SIZE
            ]

            # ------------------------------------------------
            # Zero-pad B to 256 points
            # ------------------------------------------------

            B_padded = np.zeros(NTT_SIZE, dtype=np.int64)

            B_padded[:BLOCK_SIZE] = B_block

            # ------------------------------------------------
            # 256-point NTT
            # ------------------------------------------------

            B_ntt = iterative_ntt_dif(
                B_padded.copy(),
                q,
                W
            )

            # ------------------------------------------------
            # Point-wise multiplication
            # ------------------------------------------------

            C_ntt = (
                A_ntt * B_ntt
            ) % q

            # ------------------------------------------------
            # 256-point inverse NTT
            # ------------------------------------------------

            C_block = iterative_intt_dit(
                C_ntt.copy(),
                q,
                W
            )

            # ------------------------------------------------
            # Location of this block pair in global output
            # ------------------------------------------------

            output_start = (
                block_i + block_j
            ) * BLOCK_SIZE

            # ------------------------------------------------
            # Accumulate all 256 samples
            # with global circular wrapping
            # ------------------------------------------------

            for k in range(NTT_SIZE):

                output_index = (
                    output_start + k
                ) % N

                result[output_index] += C_block[k]

                # Keep result modulo q
                result[output_index] %= q

    return result

In [165]:
hw_golden_ref = partitioned_circular_convolution_hw_mimic(a, b)


Stage 0
Length: 256
wlen: 3844
Twiddle factors: [1, 3844, 5773, 1003, 7351, 6526, 7479, 6974, 1366, 4781, 5212, 2880, 2399, 4556, 584, 2044, 7154, 1996, 6986, 1408, 4928, 1886, 6601, 3901, 2132, 7462, 3074, 3078, 3092, 3141, 7153, 5833, 1213, 405, 5258, 3041, 6803, 4608, 766, 2681, 5543, 198, 693, 6266, 6569, 3789, 1740, 6090, 5953, 1633, 1875, 2722, 1846, 6461, 3411, 417, 5300, 3188, 3477, 648, 2268, 257, 4740, 1228, 4298, 7362, 2724, 1853, 2645, 5417, 7438, 2990, 2784, 2063, 3380, 4149, 3000, 2819, 6026, 5729, 849, 6812, 799, 6637, 4027, 2573, 5165, 6556, 7584, 3501, 732, 2562, 1286, 4501, 4232, 7131, 5756, 4784, 1382, 4837, 5408, 3566, 4800, 1438, 5033, 6094, 5967, 1682, 5887, 1402, 4907, 5653, 583, 5881, 1381, 993, 7316, 2563, 5130, 2593, 5235, 6801, 4601, 4582, 675, 6203, 2508, 1097]

Stage 1
Length: 128
wlen: 5773
Twiddle factors: [1, 5773, 7351, 7479, 1366, 5212, 2399, 584, 7154, 6986, 4928, 6601, 2132, 3074, 3092, 7153, 1213, 5258, 6803, 766, 5543, 693, 6569, 1740, 5953, 1875,

In [166]:
hw_golden_ref 

array([1898, 1905, 1939, ..., 1947, 1926, 1928])

In [167]:
assert np.array_equal(hw_golden_ref, sw_golden_ref), "Hardware and software golden references do not match."